# Trinity TEFB MC Benchmark

**Brain Zone:** Executive / dlPFC + ACC  
**Dataset:** [`playra/trinity-cognitive-probes-tefb-mc`](https://www.kaggle.com/datasets/playra/trinity-cognitive-probes-tefb-mc)  
**Framework:** `kaggle-benchmarks` SDK  
**Track:** Trinity Executive Function Battery


In [ ]:
!pip install protobuf==5.29.6 --quiet
!pip install -q kaggle-benchmarks kaggle
import kaggle_benchmarks as kbench
print(f"[TEFB] Step 0: kbench v{kbench.__version__}")

In [ ]:
import os, re, glob
os.environ["RENDER_SUBRUNS"] = "False"
import kaggle_benchmarks as kbench
import kaggle
import pandas as pd
from dataclasses import dataclass
print("[TEFB] Step 1: Imports OK")

In [ ]:
print("[TEFB] Step 2: Downloading playra/trinity-cognitive-probes-tefb-mc")
os.makedirs('/kaggle/working/data', exist_ok=True)
kaggle.api.dataset_download_files('playra/trinity-cognitive-probes-tefb-mc',
    path='/kaggle/working/data', unzip=True)

csv_files = glob.glob('/kaggle/working/data/**/*.csv', recursive=True)
print(f"[TEFB] Available CSVs: {csv_files}")

CSV_PATH = next((f for f in csv_files if 'tefb_mc' in f.lower()), None)
if CSV_PATH is None:
    raise FileNotFoundError(f'[TEFB] tefb_mc.csv not found in {csv_files}')
print(f"[TEFB] Step 2: Using {CSV_PATH}")

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"[TEFB] Loaded: {len(df)} rows, cols={list(df.columns)}")

if 'question_type' in df.columns:
    df = df[df['question_type'] == 'mc'].copy()
print(f"[TEFB] MC rows: {len(df)}")

valid_mask = df['answer'].astype(str).str.strip().str.upper().str.match(r'^[A-D]$')
print(f"[TEFB] Answer validity: {valid_mask.mean():.1%}")
assert valid_mask.mean() > 0.95, 'Answer validity below 95%'
assert df['choices'].isna().sum() == 0, 'Null choices found'

dist = df['answer'].value_counts(normalize=True).sort_index()
print(f"[TEFB] Answer dist: {dict(dist.round(3))}")

eval_df = df[['question', 'choices', 'answer']].rename(
    columns={'answer': 'expected_answer'}).reset_index(drop=True)
print(f"[TEFB] Step 3: eval_df ready, {len(eval_df)} questions")

In [ ]:
@dataclass
class MCAnswer:
    answer: str
print("[TEFB] Step 4: MCAnswer schema defined")

In [ ]:
@kbench.task(name='tefb_single_mc', store_task=False)
def tefb_single_mc(llm, question: str, choices: str,
                   expected_answer: str) -> bool:
    prompt = f"""{question}

{choices}

Answer with ONLY ONE letter (A, B, C, or D).
Do not explain. Return exactly one character."""
    resp = llm.prompt(prompt, schema=MCAnswer)
    got = resp.answer.strip().upper()[:1]
    exp = str(expected_answer).strip().upper()[:1]
    return got == exp

print("[TEFB] Step 5: inner task defined")

In [ ]:
@kbench.task(name='tefb_mc_benchmark',
             description='Executive Function Battery')
def tefb_mc_benchmark(llm) -> float:
    with kbench.client.enable_cache():
        runs = tefb_single_mc.evaluate(
            llm=[llm],
            evaluation_data=eval_df,
            n_jobs=2, timeout=180,
            max_attempts=1, remove_run_files=True)
    df_res = runs.as_dataframe()
    valid = df_res[df_res['result'].notna()]
    correct = int(valid['result'].sum())
    total = len(eval_df)
    acc = float(valid['result'].mean()) if len(valid) > 0 else 0.0
    print(f'[TEFB] Valid: {len(valid)}/{total}, Correct: {correct}, Acc: {acc:.2%}')
    kbench.assertions.assert_true(True,
        expectation=f'TEFB MC accuracy: {acc:.2%} ({correct}/{total})')
    return acc

run = tefb_mc_benchmark.run(kbench.llm)
print(f'[TEFB] MC Accuracy: {run.result:.1%}')
%choose tefb_mc_benchmark